# 01_Early Fusion Multimodal

## Objective
The objective of this notebook is to combine multimodal features using early fusion and evaluate the fused model.

## Method
Early feature-level fusion followed by machine learning evaluation.

## Input
Feature datasets from multiple biometric modalities.

## Processing Tasks

1. Inspect Sample Feature Extraction from the Three Modalities.
2. Load the Pre-Trained Early Fusion Model Bundle.
3. End-to-End Verification Pipeline Test.

## Output
- Fused multimodal feature dataset.
- Fusion model performance results.
- Saved early-fusion model artifact.

## Purpose
Build a stronger multimodal verification stage by combining complementary biometric features.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Project Root Setup
PROJECT_ROOT = Path('.').resolve().parents[1]
SRC_PATH = PROJECT_ROOT / 'src' / 'Early_Fusion'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.environ.setdefault('YOLO_CONFIG_DIR', str(PROJECT_ROOT / '.cache' / 'yolo'))

from extractors import (
    extract_face_features,
    extract_voice_features,
    extract_fingerprint_features,
    fuse_features
)

print('Early Fusion environment configured successfully!')

## 1. Inspect Sample Feature Extraction from the Three Modalities
We extract features from sample files of `User_01` across Face, Voice, and Fingerprint.


In [ ]:
face_sample = list((PROJECT_ROOT / 'Dataset' / 'YOLO_Experiment' / 'Face_Classification' / 'train' / 'User_01').glob('*.png'))[0]
voice_sample = list((PROJECT_ROOT / 'Dataset' / 'Processed' / 'Voice' / 'Audio_Standardized' / 'User_01').glob('*.wav'))[0]
fp_sample = list((PROJECT_ROOT / 'Dataset' / 'Processed' / 'Fingerprint' / 'Fingerprint_CLAHE' / 'User_01').glob('*.BMP'))[0]

face_feat = extract_face_features(face_sample)
voice_feat = extract_voice_features(voice_sample)
fp_feat = extract_fingerprint_features(fp_sample)
fused_sample = fuse_features(face_feat, voice_feat, fp_feat)

print(f'Face Feature Vector Shape        : {face_feat.shape} (256D)')
print(f'Voice Feature Vector Shape       : {voice_feat.shape} (30D)')
print(f'Fingerprint Feature Vector Shape : {fp_feat.shape} (4366D)')
print(f'Fused Multimodal Vector Shape    : {fused_sample.shape} (4652D)')

## 2. Load the Pre-Trained Early Fusion Model Bundle
We inspect the trained SVM bundle from `Models/Early_Fusion/early_fusion_model.joblib`.


In [ ]:
model_path = PROJECT_ROOT / 'Models' / 'Early_Fusion' / 'early_fusion_model.joblib'
bundle = joblib.load(model_path)

print('Model Type : ', type(bundle['model']).__name__)
print('Kernel     : ', bundle['model'].kernel)
print('Classes    : ', bundle['classes'])
print('Feature Dim: ', bundle['feature_dim'])
print('Modalities : ', bundle['modality_dims'])

## 3. End-to-End Verification Pipeline Test
We pass the fused representation into the model to predict identity and confidence scores.


In [ ]:
fused_input = fused_sample.reshape(1, -1)
scaled_input = bundle['scaler'].transform(fused_input)

prediction = bundle['model'].predict(scaled_input)[0]
probs = bundle['model'].predict_proba(scaled_input)[0]

print('================ RESULTS ================')
print(f'Predicted Identity : {prediction}')
for cls_name, p in zip(bundle['classes'], probs):
    print(f'Probability [{cls_name}]: {p:.2%}')
print('=========================================')